# 07 — Ranking hybride

**Objectif :** fusionner historique des clics, TF-IDF, embeddings et popularité.

**Entrées :** artefacts des notebooks 04 à 06.  
**Sorties :** configuration hybride et métriques.  
**Dépendances :** notebooks 04, 05 et 06.  
**Temps estimé :** 1 à 3 minutes.  
**Ressources :** CUDA pour les requêtes sémantiques, FAISS CPU.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import faiss
import joblib
import numpy as np
import pandas as pd
import torch
import yaml
from sentence_transformers import SentenceTransformer


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Racine du projet introuvable.")


ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.metrics import evaluate_rankings
from src.preprocessing import normalize_query, normalize_text

CONFIG = yaml.safe_load((ROOT / "configs" / "default.yaml").read_text(encoding="utf-8"))
TOP_K = int(CONFIG["project"]["top_k"])
WEIGHTS = {
    "exact": float(CONFIG["ranking"]["exact_click_weight"]),
    "word": float(CONFIG["ranking"]["word_tfidf_weight"]),
    "char": float(CONFIG["ranking"]["char_tfidf_weight"]),
    "semantic": float(CONFIG["ranking"]["semantic_weight"]),
    "popularity": float(CONFIG["ranking"]["popularity_weight"]),
}
assert abs(sum(WEIGHTS.values()) - 1.0) < 1e-9

PROCESSED_DIR = ROOT / CONFIG["paths"]["processed_data"]
ARTIFACTS_DIR = ROOT / CONFIG["paths"]["artifacts"]
REPORTS_DIR = ROOT / CONFIG["paths"]["reports"]

products = pd.read_csv(PROCESSED_DIR / "products_clean.csv", dtype={"sku": str}).fillna("")
fit_frame = pd.read_csv(PROCESSED_DIR / "fit_clicks.csv", dtype={"sku": str}).fillna("")
validation = pd.read_csv(PROCESSED_DIR / "validation_clicks.csv", dtype={"sku": str}).fillna("")
sku_to_position = {sku: position for position, sku in enumerate(products["sku"].astype(str))}

word_vectorizer = joblib.load(ARTIFACTS_DIR / "word_tfidf.joblib")
char_vectorizer = joblib.load(ARTIFACTS_DIR / "char_tfidf.joblib")
word_matrix = joblib.load(ARTIFACTS_DIR / "word_product_matrix.joblib")
char_matrix = joblib.load(ARTIFACTS_DIR / "char_product_matrix.joblib")
semantic_index = faiss.read_index(str(ARTIFACTS_DIR / "semantic.index"))
semantic_embeddings = semantic_index.reconstruct_n(0, semantic_index.ntotal)

device = "cuda" if torch.cuda.is_available() else "cpu"
semantic_model = SentenceTransformer(CONFIG["semantic"]["model_name"], device=device)

## Préparation des signaux historiques

In [ ]:
popularity_counts = fit_frame["sku"].value_counts()
max_popularity = max(float(popularity_counts.max()), 1.0)
popularity = np.array(
    [float(popularity_counts.get(sku, 0)) / max_popularity for sku in products["sku"]],
    dtype="float32",
)

history: dict[str, dict[str, int]] = {}
for query_key, group in fit_frame.groupby("query_key"):
    history[str(query_key)] = group["sku"].value_counts().astype(int).to_dict()


def score_query(query: str) -> pd.DataFrame:
    query_text = normalize_text(query)
    query_key = normalize_query(query)
    word = (word_vectorizer.transform([query_text]) @ word_matrix.T).toarray()[0]
    char = (char_vectorizer.transform([query_text]) @ char_matrix.T).toarray()[0]
    query_embedding = semantic_model.encode(
        [query_text], normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False
    ).astype("float32")
    semantic = (query_embedding @ semantic_embeddings.T)[0]

    exact = np.zeros(len(products), dtype="float32")
    click_counts = history.get(query_key, {})
    max_clicks = max(click_counts.values(), default=1)
    for sku, count in click_counts.items():
        position = sku_to_position.get(str(sku))
        if position is not None:
            exact[position] = count / max_clicks

    final = (
        WEIGHTS["exact"] * exact
        + WEIGHTS["word"] * word
        + WEIGHTS["char"] * char
        + WEIGHTS["semantic"] * semantic
        + WEIGHTS["popularity"] * popularity
    )
    result = products[["sku", "title", "category"]].copy()
    result["exact_score"] = exact
    result["word_score"] = word
    result["char_score"] = char
    result["semantic_score"] = semantic
    result["popularity_score"] = popularity
    result["score"] = final
    return result.sort_values(["score", "sku"], ascending=[False, True], kind="stable")


display(score_query("space hero shooter").head(TOP_K))

## Évaluation

In [ ]:
actual_by_query = validation.groupby("query_key")["sku"].agg(lambda values: set(map(str, values)))
query_text_by_key = validation.groupby("query_key")["query_text"].first()
predictions = [
    score_query(query_text_by_key.loc[key]).head(TOP_K)["sku"].astype(str).tolist()
    for key in actual_by_query.index
]
assert all(len(items) == len(set(items)) == TOP_K for items in predictions)

metrics = evaluate_rankings(actual_by_query.tolist(), predictions, TOP_K)
ranking_decision = {
    "lightgbm_trained": False,
    "reason": "dataset de démonstration trop petit"
    if len(fit_frame) < 500
    else "fusion pondérée retenue comme baseline de production",
}
report = {
    "model": "weighted_hybrid",
    "device": device,
    "weights": WEIGHTS,
    "ranker_decision": ranking_decision,
    **metrics,
}

(ARTIFACTS_DIR / "hybrid_config.json").write_text(
    json.dumps({"weights": WEIGHTS, "model_name": CONFIG["semantic"]["model_name"]}, indent=2),
    encoding="utf-8",
)
(REPORTS_DIR / "metrics_hybrid.json").write_text(
    json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8"
)
print(json.dumps(report, indent=2, ensure_ascii=False))

## Conclusion

Le ranking combine cinq familles de signaux avec des poids versionnés. Le ranker supervisé est
volontairement différé lorsque le volume ne permet pas une validation sérieuse.